In [1]:
import csv
import json
from pathlib import Path
from IPython.display import display

URL_CSV = (
    "https://huggingface.co/datasets/"
    "scikit-learn/iris/resolve/main/Iris.csv"
)

URL_JSON = (
    "https://huggingface.co/datasets/"
    "osunlp/HippoRAG_2/resolve/main/case_study_university.json"
)

RUTA_CSV_ORIGINAL = Path("originales/Iris.csv")
RUTA_JSON_ORIGINAL = Path("originales/case_study_university.json")

RUTA_CSV_MODIFICADO = Path("modificados/Iris_modificado.csv")
RUTA_JSON_MODIFICADO = Path(
    "modificados/case_study_university_modificado.json"
)

print("Carpeta de trabajo:", Path.cwd())
print("Origen CSV:", URL_CSV)
print("Origen JSON:", URL_JSON)

for ruta in (RUTA_CSV_ORIGINAL, RUTA_JSON_ORIGINAL):
    if not ruta.is_file():
        raise FileNotFoundError(
            f"No se encuentra {ruta}. Ejecuta el notebook desde AE2."
        )

# Conservar los bytes para comprobar que los originales no cambien.
originales_antes = {
    ruta: ruta.read_bytes()
    for ruta in (RUTA_CSV_ORIGINAL, RUTA_JSON_ORIGINAL)
}

Carpeta de trabajo: /workspaces/analisis_de_datos/AE2
Origen CSV: https://huggingface.co/datasets/scikit-learn/iris/resolve/main/Iris.csv
Origen JSON: https://huggingface.co/datasets/osunlp/HippoRAG_2/resolve/main/case_study_university.json


In [2]:
from manejador_csv import ManejadorCSV
from manejador_json import ManejadorJSON

manejador_csv = ManejadorCSV(RUTA_CSV_ORIGINAL)
manejador_json = ManejadorJSON(RUTA_JSON_ORIGINAL)

encabezados, datos_csv_originales = manejador_csv.importar()
datos_json_originales = manejador_json.importar()

print("Registros CSV:", len(datos_csv_originales))
print("Primeros tres registros CSV:")

display([
    dict(zip(encabezados, fila))
    for fila in datos_csv_originales[:3]
])

print("\nRegistros principales JSON:", len(datos_json_originales))
print(
    "Párrafos del primer registro:",
    len(datos_json_originales[0].get("paragraphs", []))
)

print("Pregunta original:")
print(datos_json_originales[0]["question"])

Registros CSV: 150
Primeros tres registros CSV:


[{'Id': '1',
  'SepalLengthCm': '5.1',
  'SepalWidthCm': '3.5',
  'PetalLengthCm': '1.4',
  'PetalWidthCm': '0.2',
  'Species': 'Iris-setosa'},
 {'Id': '2',
  'SepalLengthCm': '4.9',
  'SepalWidthCm': '3.0',
  'PetalLengthCm': '1.4',
  'PetalWidthCm': '0.2',
  'Species': 'Iris-setosa'},
 {'Id': '3',
  'SepalLengthCm': '4.7',
  'SepalWidthCm': '3.2',
  'PetalLengthCm': '1.3',
  'PetalWidthCm': '0.2',
  'Species': 'Iris-setosa'}]


Registros principales JSON: 1
Párrafos del primer registro: 43
Pregunta original:
Which Stanford University professor works on Alzheimer's?


In [3]:
cambio_csv = manejador_csv.modificar(
    indice_fila=0,
    columna="SepalLengthCm",
    nuevo_valor="5.2"
)

print("Cambio realizado en el CSV:")
display(cambio_csv)

print("Primera fila antes:")
display(dict(zip(encabezados, datos_csv_originales[0])))

print("Primera fila después:")
display(dict(zip(encabezados, manejador_csv.datos[0])))

Cambio realizado en el CSV:


{'registro': 0,
 'campo': 'SepalLengthCm',
 'valor_anterior': '5.1',
 'valor_nuevo': '5.2'}

Primera fila antes:


{'Id': '1',
 'SepalLengthCm': '5.1',
 'SepalWidthCm': '3.5',
 'PetalLengthCm': '1.4',
 'PetalWidthCm': '0.2',
 'Species': 'Iris-setosa'}

Primera fila después:


{'Id': '1',
 'SepalLengthCm': '5.2',
 'SepalWidthCm': '3.5',
 'PetalLengthCm': '1.4',
 'PetalWidthCm': '0.2',
 'Species': 'Iris-setosa'}

In [4]:
NUEVA_PREGUNTA = (
    "¿Qué profesores de la Universidad de Stanford "
    "investigan la enfermedad de Alzheimer?"
)

cambio_json = manejador_json.modificar(
    indice_registro=0,
    campo="question",
    nuevo_valor=NUEVA_PREGUNTA
)

print("Pregunta antes:")
print(datos_json_originales[0]["question"])

print("\nPregunta después:")
print(manejador_json.datos[0]["question"])

display(cambio_json)

Pregunta antes:
Which Stanford University professor works on Alzheimer's?

Pregunta después:
¿Qué profesores de la Universidad de Stanford investigan la enfermedad de Alzheimer?


{'registro': 0,
 'campo': 'question',
 'valor_anterior': "Which Stanford University professor works on Alzheimer's?",
 'valor_nuevo': '¿Qué profesores de la Universidad de Stanford investigan la enfermedad de Alzheimer?'}

In [5]:
resultado_csv = manejador_csv.guardar(RUTA_CSV_MODIFICADO)
resultado_json = manejador_json.guardar(RUTA_JSON_MODIFICADO)

print("Resultados del proceso CSV:")
display(resultado_csv)

print("\nResultados del proceso JSON:")
display(resultado_json)

Resultados del proceso CSV:


{'clase': 'ManejadorCSV',
 'archivo_importado': 'originales/Iris.csv',
 'archivo_guardado': 'modificados/Iris_modificado.csv',
 'registros_importados': 150,
 'registros_guardados': 150,
 'modificaciones': [{'registro': 0,
   'campo': 'SepalLengthCm',
   'valor_anterior': '5.1',
   'valor_nuevo': '5.2'}],
 'metodos_utilizados': ['importar', 'modificar', 'guardar']}


Resultados del proceso JSON:


{'clase': 'ManejadorJSON',
 'archivo_importado': 'originales/case_study_university.json',
 'archivo_guardado': 'modificados/case_study_university_modificado.json',
 'registros_importados': 1,
 'registros_guardados': 1,
 'modificaciones': [{'registro': 0,
   'campo': 'question',
   'valor_anterior': "Which Stanford University professor works on Alzheimer's?",
   'valor_nuevo': '¿Qué profesores de la Universidad de Stanford investigan la enfermedad de Alzheimer?'}],
 'metodos_utilizados': ['importar', 'modificar', 'guardar']}

In [6]:
with open(
    RUTA_CSV_MODIFICADO,
    mode="r",
    encoding="utf-8",
    newline=""
) as archivo:
    lector = csv.reader(archivo)
    encabezados_guardados = next(lector)
    csv_guardado = list(lector)

with open(
    RUTA_JSON_MODIFICADO,
    mode="r",
    encoding="utf-8"
) as archivo:
    # El archivo generado admite la lectura estricta habitual.
    json_guardado = json.load(archivo)

# Comparar todo el contenido guardado con los datos de las clases.
assert encabezados_guardados == encabezados
assert csv_guardado == manejador_csv.datos
assert json_guardado == manejador_json.datos

# Comprobar los cambios específicos.
indice_columna = encabezados.index("SepalLengthCm")

assert csv_guardado[0][indice_columna] == "5.2"
assert (
    csv_guardado[0][indice_columna]
    != datos_csv_originales[0][indice_columna]
)

assert json_guardado[0]["question"] == NUEVA_PREGUNTA
assert (
    json_guardado[0]["question"]
    != datos_json_originales[0]["question"]
)

# Comprobar las cantidades de registros reportadas.
assert len(csv_guardado) == resultado_csv["registros_guardados"]
assert len(json_guardado) == resultado_json["registros_guardados"]

# Verificar que los originales mantienen exactamente sus bytes.
for ruta, contenido_original in originales_antes.items():
    assert ruta.read_bytes() == contenido_original, (
        f"El archivo original fue alterado: {ruta}"
    )

print("Cambios verificados en el CSV y el JSON guardados.")
print("Los archivos originales conservan su contenido.")

Cambios verificados en el CSV y el JSON guardados.
Los archivos originales conservan su contenido.


In [7]:
from generador_txt import GeneradorResumenTXT

generador = GeneradorResumenTXT(
    resultado_csv=resultado_csv,
    resultado_json=resultado_json,
    url_csv=URL_CSV,
    url_json=URL_JSON
)

ruta_resumen = generador.generar("resumen.txt")

print("Resumen generado en:", ruta_resumen.resolve())

Resumen generado en: /workspaces/analisis_de_datos/AE2/resumen.txt


In [8]:
with open(ruta_resumen, mode="r", encoding="utf-8") as archivo:
    contenido_resumen = archivo.read()

assert URL_CSV in contenido_resumen
assert URL_JSON in contenido_resumen
assert "SepalLengthCm" in contenido_resumen
assert NUEVA_PREGUNTA in contenido_resumen

print(contenido_resumen)
print("Flujo completo ejecutado correctamente.")

RESUMEN DE LA ACTIVIDAD AE2
Generador: GeneradorResumenTXT.generar()

FORMATO: CSV
------------------------------------------------------------
URL de origen: https://huggingface.co/datasets/scikit-learn/iris/resolve/main/Iris.csv
Archivo importado: originales/Iris.csv
Archivo guardado: modificados/Iris_modificado.csv
Clase utilizada: ManejadorCSV
Metodos utilizados: ManejadorCSV.importar(), ManejadorCSV.modificar(), ManejadorCSV.guardar()
Registros importados: 150
Registros guardados: 150
Modificaciones realizadas: 1
  Registro (indice desde 0): 0
  Campo: SepalLengthCm
  Antes: '5.1'
  Despues: '5.2'

FORMATO: JSON
------------------------------------------------------------
URL de origen: https://huggingface.co/datasets/osunlp/HippoRAG_2/resolve/main/case_study_university.json
Archivo importado: originales/case_study_university.json
Archivo guardado: modificados/case_study_university_modificado.json
Clase utilizada: ManejadorJSON
Metodos utilizados: ManejadorJSON.importar(), Manejad

### Modificaciones realizadas

Los archivos originales se descargaron manualmente desde Hugging Face.
Sus URL específicas se conservaron en las variables `URL_CSV` y
`URL_JSON` y se incluyeron en el resumen generado.

En el CSV se actualizó el campo `SepalLengthCm` de la primera fila:
su valor pasó de `5.1` a `5.2`.

En el JSON se actualizó el campo `question` del primer registro,
sustituyendo la pregunta original por una versión en español.

Ambas modificaciones se realizaron mediante métodos de sus respectivas
clases. Los resultados se guardaron con nombres nuevos en la carpeta
`modificados`.

Se volvieron a leer los archivos guardados para verificar los cambios.
También se comprobó que los originales conservaran sus bytes.

Finalmente, `GeneradorResumenTXT` recibió los resultados de ambos
procesos y las URL de origen para generar `resumen.txt`.

Cada elemento de la lista principal del JSON se cuenta como un registro.
Los párrafos anidados se conservan dentro de su registro.